In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json
import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils, experiment_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import nnsight

experiment_utils.set_seed(42)

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, width=300)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")
logger.info(f"{nnsight.__version__}")


## Load Model

In [ ]:
from src.models import ModelandTokenizer

# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)


## Load Multi-Predicate Data

In [ ]:
from src.goals.predicate_data import MultiPredicateDataset
from src.goals.predicate_sampling import MCTSSearchResults_for_TargetPredicate

# ── Select target predicate ──────────────────────────────────────────────────
PREDICATE_IDX = 0

eval_dataset = MultiPredicateDataset.load_dataset()
target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(f"Target predicate : [cyan]{target_predicate.name}[/cyan]")
rich_print(f"P+ : [bold green]{target_predicate.p_plus.description}[/bold green]  anchors={target_predicate.p_plus.anchor_tokens}")
for i, s in enumerate(target_predicate.samples):
    p_minus_desc = s.p_minus.description if s.p_minus else "(none)"
    rich_print(f"  Sample {i} — fn=[yellow]{s.function_name}[/yellow]  P-: {p_minus_desc}")


In [ ]:
model_tag = mt.name.split("/")[-1]
SAVE_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "MultiPredicate_MCTS",
    model_tag,
    f"{target_predicate.name}.json",
)

if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(
        f"No MCTS results found at {SAVE_PATH}.\n"
        "Run the MCTS sampling notebook (02.1_multi_pred_sampling) first."
    )

tp_results = MCTSSearchResults_for_TargetPredicate.load(SAVE_PATH)
rich_print(f"Loaded [bold]{len(tp_results.sample_results)}[/bold] sample results from {SAVE_PATH}")
for i, sr in enumerate(tp_results.sample_results):
    counts = [run.eval_counts for run in sr.runs]
    rich_print(f"  Sample {i} — {len(sr.runs)} runs  eval_counts={counts}")


## Collect Contrastive Pairs

In [ ]:
from src.goals.predicate_probe import (
    collect_labeled_nodes,
    sample_multipred_pairs,
    print_node_label_summary,
    print_pairs_sample,
    pairs_position_histogram,
)

# min_p_green: only keep nodes where at least this fraction of MCTS probability
# mass flows to green (correct) leaves.  Filters out nodes that are on a green
# path by luck but where the model's probability mass mostly flows to failures.
# Set to 0.0 to disable.
MIN_P_GREEN = 0.5

labeled = collect_labeled_nodes(
    tp_results=tp_results,
    tp=target_predicate,
    mt=mt,
    min_p_green=MIN_P_GREEN,
)

print_node_label_summary(labeled)


In [ ]:
# ── Form cross-tree contrastive pairs ────────────────────────────────────────
# allow_same_tree=False (default): only pair nodes from different prefill trees.
# abs_pos_tolerance=0: require exact absolute position match.
# Increase tolerance to 1-2 if prefill lengths vary by ±1 token and pair counts are low.
pairs_by_pos = sample_multipred_pairs(
    labeled_nodes=labeled,
    allow_same_tree=False,
    abs_pos_tolerance=0,
)

pairs_position_histogram(pairs_by_pos)
print_pairs_sample(pairs_by_pos, n=6)


In [ ]:
import random
from collections import defaultdict

# ── Build lookup: prefix → prompt (needed to reconstruct full input text) ────
prefix_to_prompt = {
    run.prefix: run.prompt
    for sr in tp_results.sample_results
    for run in sr.runs
}

# ── Flatten pairs into records ────────────────────────────────────────────────
# Each record: pos_text / neg_text to feed the model + metadata for analysis.
all_pairs = [pair for pairs in pairs_by_pos.values() for pair in pairs]

pair_records = []
for pair in all_pairs:
    pos_ln = pair.positive
    neg_ln = pair.negative
    pos_prompt = prefix_to_prompt.get(pos_ln.tree_id, "")
    neg_prompt = prefix_to_prompt.get(neg_ln.tree_id, "")
    pair_records.append({
        "pos_text":       pos_prompt + pos_ln.node.get_path_text(),
        "neg_text":       neg_prompt + neg_ln.node.get_path_text(),
        "abs_position":   pair.abs_position,
        "pos_depth":      pos_ln.depth,
        "neg_depth":      neg_ln.depth,
        "pos_window":     pos_ln.window,
        "neg_window":     neg_ln.window,
        "pos_tree_id":    pos_ln.tree_id,
        "neg_tree_id":    neg_ln.tree_id,
        "pos_prefill_type": pos_ln.prefill_type,
        "neg_prefill_type": neg_ln.prefill_type,
        "sample_idx":     pos_ln.sample_idx,
    })

rich_print(f"Total pair records: [bold]{len(pair_records)}[/bold]")


In [ ]:
# ── Train / test split by tree (prevents latent leakage across the split) ────
# Assign each pair based on the positive node's source tree.
# Trees (prefills) are the natural unit of independence here.
N_TRAIN_PAIRS = 10_000   # max pairs in train set; None = use all
N_TEST_PAIRS  = None     # max pairs in test set;  None = use all

random.seed(42)
all_tree_ids = list({rec["pos_tree_id"] for rec in pair_records})
random.shuffle(all_tree_ids)
n_train_trees = max(1, int(0.8 * len(all_tree_ids)))
train_trees = set(all_tree_ids[:n_train_trees])
test_trees  = set(all_tree_ids[n_train_trees:])

train_idx = [i for i, r in enumerate(pair_records) if r["pos_tree_id"] in train_trees]
test_idx  = [i for i, r in enumerate(pair_records) if r["pos_tree_id"] in test_trees]

if N_TRAIN_PAIRS is not None and len(train_idx) > N_TRAIN_PAIRS:
    train_idx = random.sample(train_idx, N_TRAIN_PAIRS)
if N_TEST_PAIRS is not None and len(test_idx) > N_TEST_PAIRS:
    test_idx = random.sample(test_idx, N_TEST_PAIRS)

n_pairs_total = len(pair_records)
rich_print(
    f"Split: [bold]{len(train_idx)}[/bold] train / [bold]{len(test_idx)}[/bold] test  "
    f"({len(train_trees)} train trees / {len(test_trees)} test trees)"
)


## Extract Latents

In [ ]:
from src.tokens import prepare_input

@torch.inference_mode()
def get_last_token_latents_batch(
    mt: ModelandTokenizer,
    texts: list[str],
    layers: list[int] | None = None,
) -> dict[int, torch.Tensor]:
    """
    Extract residual stream hidden states at the last token position for a batch of inputs.

    Simplified and batched — no interventions, grabs the last non-padding token's
    residual stream across specified layers.  Each layer stays on its native GPU.

    Args:
        mt:     ModelandTokenizer instance.
        texts:  List of input strings.
        layers: Layer indices to extract. None = all layers.

    Returns:
        Dict mapping layer_idx -> Tensor of shape (batch, hidden_dim).
    """
    if layers is None:
        layers = list(range(mt.n_layer))

    inputs = prepare_input(prompts=texts, tokenizer=mt, padding_side="left")

    layer_saves = {}
    with mt.trace(inputs, scan=False):
        for layer_idx in layers:
            layer_name = mt.layer_name_format.format(layer_idx)
            module = mt
            for attr in layer_name.split("."):
                module = module[int(attr)] if attr.isdigit() else getattr(module, attr)
            output = module.output
            if isinstance(output, tuple) or (hasattr(output, "__len__") and not hasattr(output, "shape")):
                output = output[0]
            layer_saves[layer_idx] = output[:, -1, :].save()

    return {l: layer_saves[l].float() for l in layers}


# Quick sanity check
_sample_texts = [pair_records[0]["pos_text"], pair_records[0]["neg_text"]]
_test_hs = get_last_token_latents_batch(mt, _sample_texts, layers=[0, mt.n_layer - 1])
for l, t in _test_hs.items():
    rich_print(f"Layer {l}: shape={list(t.shape)}, device={t.device}")


In [ ]:
LAYERS_TO_PROBE = list(range(mt.n_layer))
BATCH_SIZE = 32

pos_texts = [pair_records[i]["pos_text"] for i in range(n_pairs_total)]
neg_texts = [pair_records[i]["neg_text"] for i in range(n_pairs_total)]
all_texts = pos_texts + neg_texts  # 2 * n_pairs_total

n_total = len(all_texts)
rich_print(f"Extracting latents for {n_total} texts ({n_pairs_total} pairs) in batches of {BATCH_SIZE}...")

latents_by_layer: dict[int, list[torch.Tensor]] = {l: [] for l in LAYERS_TO_PROBE}

for batch_start in range(0, n_total, BATCH_SIZE):
    batch_texts = all_texts[batch_start : batch_start + BATCH_SIZE]
    if batch_start % (BATCH_SIZE * 10) == 0:
        rich_print(f"  Batch {batch_start // BATCH_SIZE + 1}/{(n_total + BATCH_SIZE - 1) // BATCH_SIZE}...")
    batch_hs = get_last_token_latents_batch(mt, batch_texts, layers=LAYERS_TO_PROBE)
    for l in LAYERS_TO_PROBE:
        latents_by_layer[l].append(batch_hs[l])

Z_pos_by_layer: dict[int, torch.Tensor] = {}
Z_neg_by_layer: dict[int, torch.Tensor] = {}

for l in LAYERS_TO_PROBE:
    all_l = torch.cat(latents_by_layer[l], dim=0)   # (2*n_pairs_total, hidden_dim)
    Z_pos_by_layer[l] = all_l[:n_pairs_total]
    Z_neg_by_layer[l] = all_l[n_pairs_total:]

rich_print(f"Done. Z_pos shape (layer 0): {list(Z_pos_by_layer[0].shape)}")


In [ ]:
# ── (Optional) Save / reload latents to avoid re-extracting ─────────────────
model_short_name = mt.name.split("/")[-1]
save_dir = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "multi_probe",
    model_short_name,
    target_predicate.name,
)
os.makedirs(save_dir, exist_ok=True)

# torch.save(Z_pos_by_layer, os.path.join(save_dir, "Z_pos_by_layer.pt"))
# torch.save(Z_neg_by_layer, os.path.join(save_dir, "Z_neg_by_layer.pt"))

# Z_pos_by_layer = torch.load(os.path.join(save_dir, "Z_pos_by_layer.pt"))
# Z_neg_by_layer = torch.load(os.path.join(save_dir, "Z_neg_by_layer.pt"))
rich_print(f"Save dir: {save_dir}")


## Analysis of the Latents
### PCA (without goal-specific engineering)

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

LAYER_FROM = 0
LAYER_TO   = mt.n_layer
LAYER_STEP = 4

pca_layers = list(range(LAYER_FROM, LAYER_TO, LAYER_STEP))
pca_results: dict[int, dict] = {}

for layer_idx in pca_layers:
    z_pos = Z_pos_by_layer[layer_idx].cpu().numpy()
    z_neg = Z_neg_by_layer[layer_idx].cpu().numpy()
    n = z_pos.shape[0]

    all_z = np.concatenate([z_pos, z_neg], axis=0)
    pca = PCA(n_components=2)
    all_2d = pca.fit_transform(all_z)

    pca_results[layer_idx] = {
        "pos_2d": all_2d[:n],
        "neg_2d": all_2d[n:],
        "explained_variance_ratio": pca.explained_variance_ratio_,
    }

rich_print(f"Computed PCA for {len(pca_layers)} layers: {pca_layers[0]}..{pca_layers[-1]} (step={LAYER_STEP})")
rich_print(f"Latents per layer: {n} pos (P+) + {n} neg (P-) = {2*n}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 7))

all_x = np.concatenate([np.concatenate([r["pos_2d"][:, 0], r["neg_2d"][:, 0]]) for r in pca_results.values()])
all_y = np.concatenate([np.concatenate([r["pos_2d"][:, 1], r["neg_2d"][:, 1]]) for r in pca_results.values()])
pad_x = (all_x.max() - all_x.min()) * 0.05
pad_y = (all_y.max() - all_y.min()) * 0.05
xlim = (all_x.min() - pad_x, all_x.max() + pad_x)
ylim = (all_y.min() - pad_y, all_y.max() + pad_y)

neg_scatter = ax.scatter([], [], alpha=0.15, s=6, c="tab:red",  label="P- pursuing")
pos_scatter = ax.scatter([], [], alpha=0.15, s=6, c="tab:blue", label="P+ pursuing")
ax.legend(markerscale=4, loc="upper right")
ax.set_xlim(xlim); ax.set_ylim(ylim)
title = ax.set_title("")

FIGURE_DIR = os.path.join(save_dir, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

model_short = mt.name.split("/")[-1]

def update(frame_idx):
    layer_idx = pca_layers[frame_idx]
    r = pca_results[layer_idx]
    evr = r["explained_variance_ratio"]
    pos_scatter.set_offsets(r["pos_2d"])
    neg_scatter.set_offsets(r["neg_2d"])
    ax.set_xlabel(f"PC1 ({evr[0]:.1%} var)")
    ax.set_ylabel(f"PC2 ({evr[1]:.1%} var)")
    title.set_text(f"Layer {layer_idx} | {model_short} | {target_predicate.name} | P+ vs P-")
    return pos_scatter, neg_scatter, title

anim = animation.FuncAnimation(fig, update, frames=len(pca_layers), interval=200, blit=False)
anim.save(os.path.join(FIGURE_DIR, "pca.gif"), writer="pillow", fps=2, dpi=120)
plt.close(fig)
HTML(anim.to_jshtml())


## Train Goal Directions

In [ ]:
import numpy as np

n_pairs = n_pairs_total
n_layers = len(LAYERS_TO_PROBE)

rich_print(f"Train pairs: {len(train_idx)}, Test pairs: {len(test_idx)}")

# ── Baseline: mean difference direction ─────────────────────────────────────
goal_directions: dict[int, torch.Tensor] = {}
train_accuracies: dict[int, float] = {}
test_accuracies: dict[int, float] = {}

for layer_idx in LAYERS_TO_PROBE:
    z_pos_l = Z_pos_by_layer[layer_idx]
    z_neg_l = Z_neg_by_layer[layer_idx]

    z_pos_train = z_pos_l[train_idx]
    z_neg_train = z_neg_l[train_idx]

    v_l = z_pos_train.mean(dim=0) - z_neg_train.mean(dim=0)
    v_l = v_l / (v_l.norm() + 1e-8)
    goal_directions[layer_idx] = v_l

    train_acc = (z_pos_train @ v_l > z_neg_train @ v_l).float().mean().item()
    train_accuracies[layer_idx] = train_acc

    z_pos_test = z_pos_l[test_idx]
    z_neg_test = z_neg_l[test_idx]
    test_acc = (z_pos_test @ v_l > z_neg_test @ v_l).float().mean().item()
    test_accuracies[layer_idx] = test_acc

rich_print(f"\n{'Layer':>6} {'Train':>8} {'Test':>8} {'Device':>12}")
rich_print(f"{'-----':>6} {'-----':>8} {'-----':>8} {'------':>12}")
for l in LAYERS_TO_PROBE:
    marker = " [bold green]<--[/bold green]" if test_accuracies[l] >= 0.8 else ""
    rich_print(f"{l:>6} {train_accuracies[l]:>8.3f} {test_accuracies[l]:>8.3f} {str(z_pos_l.device):>12}{marker}")


In [ ]:
# ── Contrastive objective (Schut et al. Section 4.1.2) ──────────────────────
# min  λ₁||v||₁ + λ₂||v||₂² + (1/N) Σᵢ max(0, margin - v·(z⁺ᵢ - z⁻ᵢ))

@torch.enable_grad()
def learn_contrastive_direction(
    z_pos: torch.Tensor,
    z_neg: torch.Tensor,
    l1_lambda: float = 0.0,
    l2_lambda: float = 1e-3,
    margin: float = 1.0,
    lr: float = 1e-2,
    n_steps: int = 1000,
    patience: int = 50,
) -> tuple[torch.Tensor, dict]:
    device = z_pos.device
    hidden_dim = z_pos.shape[1]
    delta = z_pos - z_neg

    v = torch.randn(hidden_dim, device=device, dtype=z_pos.dtype) * 0.01
    v.requires_grad_(True)
    optimizer = torch.optim.Adam([v], lr=lr)

    best_loss = float("inf")
    patience_counter = 0
    final_step = n_steps

    for step in range(n_steps):
        optimizer.zero_grad()
        margins = margin - (delta @ v)
        hinge = torch.clamp(margins, min=0).mean()
        l1 = l1_lambda * v.abs().mean()
        l2 = l2_lambda * (v ** 2).mean()
        loss = hinge + l1 + l2
        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                final_step = step + 1
                break

    with torch.no_grad():
        v_norm = v / (v.norm() + 1e-8)

    sparsity = (v_norm.abs() < 0.01).float().mean().item()
    return v_norm.detach(), {"steps": final_step, "best_loss": best_loss, "sparsity": sparsity}


In [ ]:
L1_LAMBDA = 0.0
L2_LAMBDA = 1e-3
MARGIN    = 1.0
LR        = 1e-2
N_STEPS   = 1000

contrastive_directions: dict[int, torch.Tensor] = {}
contrastive_train_accs: dict[int, float] = {}
contrastive_test_accs:  dict[int, float] = {}
contrastive_infos:      dict[int, dict]  = {}

reg_desc = []
if L1_LAMBDA > 0: reg_desc.append(f"λ₁={L1_LAMBDA}")
if L2_LAMBDA > 0: reg_desc.append(f"λ₂={L2_LAMBDA}")
if not reg_desc: reg_desc.append("no regularization")
rich_print(f"Learning contrastive directions for {n_layers} layers ({', '.join(reg_desc)}, margin={MARGIN})...")

for layer_idx in LAYERS_TO_PROBE:
    z_pos_l = Z_pos_by_layer[layer_idx]
    z_neg_l = Z_neg_by_layer[layer_idx]

    v_l, info = learn_contrastive_direction(
        z_pos_l[train_idx], z_neg_l[train_idx],
        l1_lambda=L1_LAMBDA, l2_lambda=L2_LAMBDA,
        margin=MARGIN, lr=LR, n_steps=N_STEPS,
    )
    contrastive_directions[layer_idx] = v_l
    contrastive_infos[layer_idx] = info

    contrastive_train_accs[layer_idx] = (
        (z_pos_l[train_idx] @ v_l > z_neg_l[train_idx] @ v_l).float().mean().item()
    )
    z_pos_test = z_pos_l[test_idx]
    z_neg_test = z_neg_l[test_idx]
    contrastive_test_accs[layer_idx] = (
        (z_pos_test @ v_l > z_neg_test @ v_l).float().mean().item()
    )

    if layer_idx % 10 == 0:
        rich_print(
            f"  Layer {layer_idx:>3}: test_acc={contrastive_test_accs[layer_idx]:.3f}, "
            f"steps={info['steps']}, sparsity={info['sparsity']:.3f}"
        )


In [ ]:
import matplotlib.pyplot as plt

layers_sorted = sorted(LAYERS_TO_PROBE)
md_test = [test_accuracies[l]           for l in layers_sorted]
ct_test = [contrastive_test_accs[l]     for l in layers_sorted]
ct_sparsity = [contrastive_infos[l]["sparsity"] for l in layers_sorted]
reg_label = ", ".join(reg_desc)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

ax = axes[0]
ax.plot(layers_sorted, md_test, label="Mean Difference (baseline)", alpha=0.8, linewidth=1.5)
ax.plot(layers_sorted, ct_test, label=f"Contrastive ({reg_label}, m={MARGIN})", alpha=0.8, linewidth=1.5)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Chance")
ax.set_ylabel("Test Constraint Accuracy (v·z⁺ > v·z⁻)")
ax.set_title(
    f"Goal Direction: Mean Diff vs Contrastive | {model_short_name} | "
    f"{target_predicate.name} | {n_pairs} pairs"
)
ax.legend(loc="lower right")
ax.set_ylim(0.4, 1.05)

ax2 = axes[1]
ax2.bar(layers_sorted, ct_sparsity, alpha=0.6, color="tab:green", width=0.8)
ax2.set_xlabel("Layer")
ax2.set_ylabel("Sparsity\n(frac |vᵢ|<0.01)")
ax2.set_ylim(0, 1)
ax2.set_xlim(layers_sorted[0] - 0.5, layers_sorted[-1] + 0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "direction_accuracy.png"), dpi=150)
plt.show()


### Contrastive Direction Projection (Goal Axis + Residual PCA)

In [ ]:
from sklearn.metrics import roc_auc_score

LAYER_FROM_CD = 0
LAYER_TO_CD   = mt.n_layer
LAYER_STEP_CD = 1
DO_NORMALIZE  = True

cd_layers = list(range(LAYER_FROM_CD, LAYER_TO_CD, LAYER_STEP_CD))
test_idx_arr = np.array(test_idx)

cd_proj_results: dict[int, dict] = {}

rich_print(f"\n{'Layer':>6} {'AUC(goal)':>10} {'AUC(resid)':>11} {'ResidVar%':>10}")
rich_print(f"{'-----':>6} {'---------':>10} {'----------':>11} {'---------':>10}")

for layer_idx in cd_layers:
    v = contrastive_directions[layer_idx].cpu().numpy()

    z_pos = Z_pos_by_layer[layer_idx][test_idx_arr].cpu().numpy()
    z_neg = Z_neg_by_layer[layer_idx][test_idx_arr].cpu().numpy()
    n_test = z_pos.shape[0]

    all_z = np.concatenate([z_pos, z_neg], axis=0)
    if DO_NORMALIZE:
        norms = np.maximum(np.linalg.norm(all_z, axis=1, keepdims=True), 1e-8)
        all_z = all_z / norms

    proj_goal = all_z @ v
    z_residual = all_z - np.outer(proj_goal, v)

    pca_res = PCA(n_components=1)
    proj_residual = pca_res.fit_transform(z_residual).squeeze()

    labels = np.concatenate([np.ones(n_test), np.zeros(n_test)])
    auc_goal  = roc_auc_score(labels, proj_goal)
    auc_resid = roc_auc_score(labels, proj_residual)

    cd_proj_results[layer_idx] = {
        "pos_goal":       proj_goal[:n_test],
        "neg_goal":       proj_goal[n_test:],
        "pos_residual":   proj_residual[:n_test],
        "neg_residual":   proj_residual[n_test:],
        "auc_goal":       auc_goal,
        "auc_residual":   auc_resid,
        "residual_var_ratio": pca_res.explained_variance_ratio_[0],
    }

    if layer_idx % 8 == 0:
        rich_print(f"{layer_idx:>6} {auc_goal:>10.4f} {auc_resid:>11.4f} {pca_res.explained_variance_ratio_[0]:>10.1%}")


In [ ]:
# ── AUC summary plot ──────────────────────────────────────────────────────────
layers_sorted_cd = sorted(cd_proj_results.keys())
auc_goal_vals  = [cd_proj_results[l]["auc_goal"]      for l in layers_sorted_cd]
auc_resid_vals = [cd_proj_results[l]["auc_residual"]  for l in layers_sorted_cd]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(layers_sorted_cd, auc_goal_vals,  label="AUC on Goal Axis (P+ vs P-)", linewidth=1.5)
ax.plot(layers_sorted_cd, auc_resid_vals, label="AUC on Residual PC1", linewidth=1.5, alpha=0.7)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("ROC-AUC (test set)")
ax.set_title(
    f"P+ vs P- Separability on Contrastive Goal Direction | "
    f"{mt.name.split('/')[-1]} | {target_predicate.name}"
)
ax.legend(loc="lower right")
ax.set_ylim(0.4, 1.0)
ax.set_xlim(layers_sorted_cd[0], layers_sorted_cd[-1])
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "auc_summary.png"), dpi=150)
plt.show()


## Save Directions

In [ ]:
torch.save(contrastive_directions, os.path.join(save_dir, "contrastive_directions.pt"))
torch.save(goal_directions,        os.path.join(save_dir, "mean_diff_directions.pt"))

with open(os.path.join(save_dir, "contrastive_infos.json"), "w") as f:
    json.dump(contrastive_infos, f, indent=2)

with open(os.path.join(save_dir, "test_accuracies.json"), "w") as f:
    json.dump({"mean_diff": test_accuracies, "contrastive": contrastive_test_accs}, f, indent=2)

rich_print(f"Saved directions to [bold]{save_dir}[/bold]")


## Test Causality (Steering)

Steer the model with the learned P+ direction and check whether it shifts generation
toward implementing P+ first.  Since all orderings are correct (green), we evaluate
by checking which predicate's anchor token appears first in the steered generation —
not by unit tests.

`coeff > 0` steers toward P+ first; `coeff < 0` steers toward P- first.

In [ ]:
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch, get_hs, PatchSpec
from src.tokens import prepare_input

# ── Select sample + build prompt ─────────────────────────────────────────────
SAMPLE_IDX = 0
sample = target_predicate.samples[SAMPLE_IDX]

# Use an inconsequential (unprefilled) prompt to see the model's natural preference
prefill = ""
if supports_chat_template(mt):
    prompt = apply_chat_template(
        mt=mt,
        user_query=sample.instruction.strip() if hasattr(sample, "instruction") else "",
        incomplete_code=sample.code + prefill,
    )
else:
    prompt = sample.code + prefill

rich_print(f"Prompt (last 200 chars):\n...{prompt[-200:]}")


In [ ]:
# ── Extract hidden state at last position for norm-scaling ───────────────────
tokenized = prepare_input(prompts=prompt, tokenizer=mt)
hs = get_hs(
    mt=mt,
    input=tokenized,
    locations=[(mt.layer_name_format.format(l), -1) for l in range(mt.n_layer)],
)


In [ ]:
# ── Load directions + build steering patches ─────────────────────────────────
loaded_directions = torch.load(
    os.path.join(save_dir, "contrastive_directions.pt"),
    map_location="cuda:0",
)

# Steer layers where test accuracy is highest — inspect auc_summary.png to pick range
layer_from = 30
layer_to   = 45
coeff      = 0.08  # positive = toward P+; negative = toward P-

steer_patches = []
for layer_idx in range(layer_from, layer_to):
    layer_name = mt.layer_name_format.format(layer_idx)
    h_orig   = hs[(layer_name, -1)]
    goal_dir = loaded_directions[layer_idx].to(h_orig.device)
    scaled   = goal_dir * coeff * (h_orig.norm() / goal_dir.norm())
    steer_patches.append(
        PatchSpec(location=(layer_name, -1), patch=scaled, strategy="add", on_pass="all")
    )

rich_print(f"Steering layers {layer_from}–{layer_to - 1} with coeff={coeff}")


In [ ]:
# ── Generate baseline and steered outputs ────────────────────────────────────
N_GENERATIONS = 50
MAX_NEW_TOKENS = 64

baseline_outputs = generate_with_patch(
    mt=mt, inputs=prompt, patches=None,
    do_sample=True, n_gen_per_prompt=N_GENERATIONS,
    remove_prefix=True, max_new_tokens=MAX_NEW_TOKENS,
)

steered_outputs = generate_with_patch(
    mt=mt, inputs=prompt, patches=steer_patches,
    do_sample=True, n_gen_per_prompt=N_GENERATIONS,
    remove_prefix=True, max_new_tokens=MAX_NEW_TOKENS,
)

rich_print(f"Baseline ({N_GENERATIONS} gens):  {baseline_outputs[:3]}")
rich_print(f"Steered  ({N_GENERATIONS} gens):  {steered_outputs[:3]}")


In [ ]:
# ── Evaluate: which predicate appears first? (anchor-token scoring) ───────────
from src.goals.anchor import _extract_boolean_expr, _split_clauses

p_plus_anchors  = target_predicate.p_plus.anchor_tokens
p_minus_anchors = sample.p_minus.anchor_tokens if sample.p_minus else []

def predicate_order(text: str) -> str:
    """Return 'p_plus_first', 'p_minus_first', or 'unclear'."""
    full = sample.code + prefill + text
    result = _extract_boolean_expr(full)
    if result is None:
        # Fall back to raw anchor search
        p_plus_pos  = min((full.find(a) for a in p_plus_anchors  if full.find(a) >= 0), default=None)
        p_minus_pos = min((full.find(a) for a in p_minus_anchors if full.find(a) >= 0), default=None)
    else:
        expr, expr_start = result
        p_plus_pos  = next((expr.find(a) for a in p_plus_anchors  if expr.find(a) >= 0), None)
        p_minus_pos = next((expr.find(a) for a in p_minus_anchors if expr.find(a) >= 0), None)

    if p_plus_pos is None and p_minus_pos is None: return "unclear"
    if p_minus_pos is None:                         return "p_plus_first"
    if p_plus_pos  is None:                         return "p_minus_first"
    return "p_plus_first" if p_plus_pos <= p_minus_pos else "p_minus_first"

from collections import Counter

baseline_orders = Counter(predicate_order(g) for g in baseline_outputs)
steered_orders  = Counter(predicate_order(g) for g in steered_outputs)

rich_print(f"\nP+ anchors : {p_plus_anchors}")
rich_print(f"P- anchors : {p_minus_anchors}")
rich_print(f"\nBaseline predicate order: {dict(baseline_orders)}")
rich_print(f"Steered  predicate order: {dict(steered_orders)}")

def _rate(counter, key):
    total = sum(counter.values())
    return counter.get(key, 0) / total if total > 0 else 0.0

rich_print(
    f"\nP+ first rate: baseline={_rate(baseline_orders, 'p_plus_first'):.2f}  "
    f"steered={_rate(steered_orders, 'p_plus_first'):.2f}  "
    f"Δ={_rate(steered_orders, 'p_plus_first') - _rate(baseline_orders, 'p_plus_first'):+.2f}"
)
